# 02 · Download Qwen2.5-3B-Instruct and Llama-3.2-3B-Instruct to Google Drive

Downloads both models once (~12.6 GB total) into your project folder, so later sessions just copy them to local disk.

**Before running:**
1. Llama 3.2 is gated. Open https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct, log in and accept the license (approval is usually quick).
2. Create a Hugging Face access token (read) at https://huggingface.co/settings/tokens.
3. In Colab, click the 🔑 **Secrets** icon in the left sidebar, add a secret named `HF_TOKEN` with the token, and switch on notebook access. (If you skip this, the notebook asks for the token instead.)

A CPU runtime is fine for this notebook.

In [1]:
# ==== settings ====
PROJECT_DIR = "/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching"
MODELS = {
    # folder name in Drive           : Hugging Face repo id
    "Qwen2.5-3B-Instruct"            : "Qwen/Qwen2.5-3B-Instruct",
    "Llama-3.2-3B-Instruct"          : "meta-llama/Llama-3.2-3B-Instruct",
}
# only the files vLLM/transformers need; skips Llama's duplicate 'original/' checkpoint (~6 GB)
ALLOW = ["*.json", "*.safetensors", "tokenizer*", "*.model", "*.txt", "*.tiktoken"]
IGNORE = ["original/*", "*.pth", "*.bin", "*.gguf"]

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os

def as_token(v):
    """Colab's userdata.get() can return a plain string or a dict, depending on the Colab version."""
    if isinstance(v, dict):
        v = v.get("value") or v.get("secret") or v.get("token") or \
            next((x for x in v.values() if isinstance(x, str) and x.strip().startswith("hf_")), None)
    return v.strip() if isinstance(v, str) and v.strip() else None

HF_TOKEN, source = None, None
try:
    from google.colab import userdata
    HF_TOKEN, source = as_token(userdata.get("HF_TOKEN")), "Colab secret HF_TOKEN"
except Exception as e:
    print("Could not read the Colab secret:", type(e).__name__)
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN, source = as_token(getpass("Hugging Face token (starts with hf_): ")), "typed in"
assert HF_TOKEN, "No Hugging Face token found."
os.environ["HF_TOKEN"] = HF_TOKEN
!pip install -q -U huggingface_hub hf_transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # faster downloads

from huggingface_hub import whoami, hf_hub_download
print(f"Token from {source}; logged in to Hugging Face as:", whoami(token=HF_TOKEN)["name"])
try:
    hf_hub_download("meta-llama/Llama-3.2-3B-Instruct", "config.json", token=HF_TOKEN, local_dir="/content/_hf_check")
    print("Llama 3.2 access: OK")
except Exception as e:
    print("⚠ No access to Llama 3.2 yet - accept the license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct\n ", e)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.0.0 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.0.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.0.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.0.0 which

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Token from Colab secret HF_TOKEN; logged in to Hugging Face as: AIHero123


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

Llama 3.2 access: OK


In [4]:
import shutil, subprocess, glob
from huggingface_hub import snapshot_download

MODEL_DIR = f"{PROJECT_DIR}/models"
os.makedirs(MODEL_DIR, exist_ok=True)

def is_complete(path):
    return os.path.exists(f"{path}/config.json") and glob.glob(f"{path}/*.safetensors") and \
           (os.path.exists(f"{path}/tokenizer.json") or os.path.exists(f"{path}/tokenizer.model"))

for name, repo in MODELS.items():
    dst = f"{MODEL_DIR}/{name}"
    if is_complete(dst):
        print(f"✓ {name} already in Drive - skipping"); continue
    tmp = f"/content/_dl/{name}"                     # download to local disk, then copy to Drive (faster)
    print(f"Downloading {repo} ...")
    snapshot_download(repo_id=repo, local_dir=tmp, allow_patterns=ALLOW, ignore_patterns=IGNORE, token=HF_TOKEN)
    print(f"Copying to {dst} ...")
    shutil.copytree(tmp, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".cache"))
    shutil.rmtree(tmp)
    print(f"✓ {name} done")

✓ Qwen2.5-3B-Instruct already in Drive - skipping


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Copying to /content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/models/Llama-3.2-3B-Instruct ...
✓ Llama-3.2-3B-Instruct done


In [5]:
!du -sh "{MODEL_DIR}"/*
for name in MODELS:
    print(name, sorted(os.listdir(f"{MODEL_DIR}/{name}")))
total, used, free = shutil.disk_usage("/content/drive")
print(f"\nDrive: {used/1e9:.1f} GB used, {free/1e9:.1f} GB free")

6.0G	/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/models/Llama-3.2-3B-Instruct
5.8G	/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/models/Qwen2.5-3B-Instruct
Qwen2.5-3B-Instruct ['config.json', 'generation_config.json', 'merges.txt', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']
Llama-3.2-3B-Instruct ['LICENSE.txt', 'config.json', 'generation_config.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']

Drive: 39.7 GB used, 67.7 GB free


Done. Next: **03_run_experiments.ipynb** (switch the runtime to a GPU first).